# 🎹 Rishky Piano — Interactive Notebook

A state-of-the-art interactive piano playable with your **MacBook keyboard**.  
Built with Web Audio API for realistic sound synthesis.

---

## Keyboard Layout

```
 Black:   W  E     T  Y  U     O  P  [
 White:  A  S  D  F  G  H  J  K  L  ;  '  [
         C  D  E  F  G  A  B  C  D  E  F  F# ...
              (main octave)    (octave above)

 Lower:  Z  X  C  V  B  N  M
         C  D  E  F  G  A  B  (octave below)

 SPACE = Sustain | ↑/↓ = Shift octave
```

## Sound Engine
- Multi-harmonic additive synthesis
- Piano ADSR envelope (fast attack, long decay, low sustain)
- Room reverb via convolution
- Stereo spread (low notes left, high notes right)
- Dynamics compression
- 5 instrument tones: Piano, Electric Piano, Organ, Strings, Bells

## How to run
Run the cell below to launch the piano inline, or run `streamlit run app.py` for the full-screen experience.

In [ ]:
from IPython.display import HTML, display
from pathlib import Path

# Load the piano HTML
piano_html = Path('piano.html').read_text(encoding='utf-8')

# Display inline in the notebook
display(HTML(f"""
<div style='width:100%; height:900px; overflow:auto; border-radius:12px;'>
  {piano_html}
</div>
"""))


---

## Audio Synthesis Deep Dive

The piano sound is built from layered sine waves (additive synthesis).  
Each note triggers a bundle of oscillators tuned to the harmonic series:

| Harmonic | Ratio | Relative Amplitude |
|---------|-------|-------------------|
| 1st (fundamental) | 1× | 100% |
| 2nd | 2× | 50% |
| 3rd | 3× | 25% |
| 4th | 4× | 12% |
| 6th | 6× | 6% |
| 8th | 8× | 3% |

### ADSR Envelope (Piano profile)

```
Volume
 │         Attack (4ms)
 │        /\
 │       /  \  Decay (800ms)
 │      /    \
 │     /      \_____  Sustain (15%)
 │    /             \
 │   /               \ Release (1.2s)
 │  /                 \
 │_/                   \____
 └─────────────────────────→ Time
```

### MIDI Note → Frequency Formula

$$f = 440 \times 2^{(n - 69) / 12}$$

Where $n$ is the MIDI note number (A4 = 69, middle C = 60).

In [ ]:
import math
import numpy as np

# Reproduce the frequency formula in Python
def midi_to_freq(midi_note: int) -> float:
    """Convert MIDI note number to frequency in Hz."""
    return 440.0 * (2 ** ((midi_note - 69) / 12))

NOTE_NAMES = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']

def midi_to_name(midi_note: int) -> str:
    octave = midi_note // 12 - 1
    note = NOTE_NAMES[midi_note % 12]
    return f"{note}{octave}"

# Print the standard piano range
print(f"{'Note':<6} {'MIDI':>4} {'Frequency':>10}")
print("-" * 24)
for midi in [48, 52, 55, 60, 64, 67, 69, 72, 76, 84]:
    print(f"{midi_to_name(midi):<6} {midi:>4} {midi_to_freq(midi):>10.2f} Hz")


In [ ]:
# Chord detection (mirrors the JavaScript logic)
def detect_chord(midi_notes: list[int]) -> str | None:
    """Detect chord name from a list of MIDI note numbers."""
    classes = sorted(set(m % 12 for m in midi_notes))
    root = classes[0]
    intervals = tuple(sorted((c - root) % 12 for c in classes))
    
    chord_types = {
        (0,4,7):      'Major',
        (0,3,7):      'Minor',
        (0,4,7,11):   'Maj7',
        (0,3,7,10):   'min7',
        (0,4,7,10):   'Dom7',
        (0,3,6):      'dim',
        (0,4,8):      'Aug',
        (0,2,7):      'sus2',
        (0,5,7):      'sus4',
    }
    chord_type = chord_types.get(intervals)
    if chord_type:
        root_name = NOTE_NAMES[root]
        return f"{root_name} {chord_type}"
    return None

# Test some chords
test_chords = [
    [60, 64, 67],       # C E G  → C Major
    [60, 63, 67],       # C Eb G → C Minor
    [60, 64, 67, 71],   # C E G B → C Maj7
    [69, 72, 76],       # A C# E → A Major
    [62, 65, 69],       # D F A → D Minor
]

for notes in test_chords:
    names = [midi_to_name(n) for n in notes]
    chord = detect_chord(notes)
    print(f"{' + '.join(names):<25} → {chord}")


---
## Running the Streamlit App

```bash
# Install dependencies
pip install streamlit

# Run locally
streamlit run app.py

# Or open piano.html directly in your browser for zero-dependency use
open piano.html
```

## Deploying to Streamlit Community Cloud

1. Push this repo to GitHub
2. Go to [share.streamlit.io](https://share.streamlit.io)
3. Connect your GitHub account and select this repo
4. Set `app.py` as the main file — done! Free permanent hosting.